In [1]:
# Bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use('seaborn')

C:\Users\eduar\AppData\Local\Temp\ipykernel_4072\409872213.py:10: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn')


In [79]:
# Localiza a raiz do projeto independentemente da máquina ou diretório de execução.
PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").exists()
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

caminho_dados = RAW_DATA_DIR / "application_train.csv"
bureau_balance = RAW_DATA_DIR / "bureau_balance.csv"
bureau = RAW_DATA_DIR / "bureau.csv"
credit_card_balance = RAW_DATA_DIR / "credit_card_balance.csv"
installments_payment = RAW_DATA_DIR / "installments_payments.csv"
pos_cash_balance = RAW_DATA_DIR / "POS_CASH_balance.csv"
previous_application = RAW_DATA_DIR / "previous_application.csv"

In [80]:
dados = pd.read_csv(caminho_dados)
bureau = pd.read_csv(bureau)
bureau_balance = pd.read_csv(bureau_balance)
previous_application = pd.read_csv(previous_application)
pos_cash_balance = pd.read_csv(pos_cash_balance)
credit_card_balance = pd.read_csv(credit_card_balance)
installments_payment = pd.read_csv(installments_payment)

# ==========================================================
# BUREAU_BALANCE -> agrega por SK_ID_BUREAU
# ==========================================================

bureau_balance_agg = (
    bureau_balance
    .groupby("SK_ID_BUREAU")
    .agg(
        MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
        MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
        MONTHS_BALANCE_COUNT=("MONTHS_BALANCE", "count"),
        STATUS_NUNIQUE=("STATUS", "nunique"),
    )
    .reset_index()
)

bureau = bureau.merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left",
)

# ==========================================================
# BUREAU -> agrega por cliente
# ==========================================================

bureau_agg = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        TOTAL_CREDITOS_BUREAU=("SK_ID_BUREAU", "count"),
        CREDITO_ATIVO=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
        CREDITO_FECHADO=("CREDIT_ACTIVE", lambda x: (x == "Closed").sum()),
        DIAS_CREDITO_MEDIO=("DAYS_CREDIT", "mean"),
        VALOR_CREDITO_TOTAL=("AMT_CREDIT_SUM", "sum"),
        DIVIDA_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
        LIMITE_TOTAL=("AMT_CREDIT_SUM_LIMIT", "sum"),
        ATRASO_MAXIMO=("CREDIT_DAY_OVERDUE", "max"),
        CONSULTAS_BUREAU=("CNT_CREDIT_PROLONG", "sum"),
        HISTORICO_MESES=("MONTHS_BALANCE_COUNT", "sum"),
    )
    .reset_index()
)

# ==========================================================
# PREVIOUS APPLICATION
# ==========================================================

previous_application_agg = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        TOTAL_SOLICITACOES=("SK_ID_PREV", "count"),
        CREDITO_SOLICITADO_TOTAL=("AMT_APPLICATION", "sum"),
        CREDITO_APROVADO_TOTAL=("AMT_CREDIT", "sum"),
        ANUIDADE_MEDIA=("AMT_ANNUITY", "mean"),
        DIAS_DECISAO_MEDIO=("DAYS_DECISION", "mean"),
    )
    .reset_index()
)

# ==========================================================
# POS CASH
# ==========================================================

pos_cash_balance_agg = (
    pos_cash_balance
    .groupby("SK_ID_CURR")
    .agg(
        TOTAL_POS=("SK_ID_PREV", "count"),
        DPD_MAX=("SK_DPD", "max"),
        DPD_DEF_MAX=("SK_DPD_DEF", "max"),
        MESES_POS=("MONTHS_BALANCE", "count"),
    )
    .reset_index()
)

# ==========================================================
# CREDIT CARD
# ==========================================================

credit_card_balance_agg = (
    credit_card_balance
    .groupby("SK_ID_CURR")
    .agg(
        TOTAL_CARTOES=("SK_ID_PREV", "nunique"),
        LIMITE_CARTAO_TOTAL=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
        SALDO_TOTAL=("AMT_BALANCE", "sum"),
        SAQUE_TOTAL=("AMT_DRAWINGS_CURRENT", "sum"),
        PAGAMENTO_TOTAL=("AMT_PAYMENT_TOTAL_CURRENT", "sum"),
        DPD_CARTAO_MAX=("SK_DPD", "max"),
    )
    .reset_index()
)

# ==========================================================
# INSTALLMENTS
# ==========================================================

installments_payment_agg = (
    installments_payment
    .groupby("SK_ID_CURR")
    .agg(
        TOTAL_PARCELAS=("SK_ID_PREV", "count"),
        VALOR_PAGO_TOTAL=("AMT_PAYMENT", "sum"),
        VALOR_PARCELAS_TOTAL=("AMT_INSTALMENT", "sum"),
        ATRASO_PAGAMENTO_MAX=("DAYS_ENTRY_PAYMENT", "max"),
    )
    .reset_index()
)

# ==========================================================
# MERGES
# ==========================================================

dados = (
    dados
    .merge(bureau_agg, on="SK_ID_CURR", how="left")
    .merge(previous_application_agg, on="SK_ID_CURR", how="left")
    .merge(pos_cash_balance_agg, on="SK_ID_CURR", how="left")
    .merge(credit_card_balance_agg, on="SK_ID_CURR", how="left")
    .merge(installments_payment_agg, on="SK_ID_CURR", how="left")
)

print(dados.shape)
dados.head()

(307511, 151)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_CARTOES,LIMITE_CARTAO_TOTAL,SALDO_TOTAL,SAQUE_TOTAL,PAGAMENTO_TOTAL,DPD_CARTAO_MAX,TOTAL_PARCELAS,VALOR_PAGO_TOTAL,VALOR_PARCELAS_TOTAL,ATRASO_PAGAMENTO_MAX
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,19.0,219625.695,219625.695,-49.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,25.0,1618864.650,1618864.650,-544.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,21288.465,21288.465,-727.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,1.0,1620000.0,0.0,0.0,0.0,0.0,16.0,1007153.415,1007153.415,-12.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,66.0,806127.975,835985.340,-14.0


In [81]:
dados.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_CARTOES,LIMITE_CARTAO_TOTAL,SALDO_TOTAL,SAQUE_TOTAL,PAGAMENTO_TOTAL,DPD_CARTAO_MAX,TOTAL_PARCELAS,VALOR_PAGO_TOTAL,VALOR_PARCELAS_TOTAL,ATRASO_PAGAMENTO_MAX
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,19.0,219625.695,219625.695,-49.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,25.0,1618864.650,1618864.650,-544.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,21288.465,21288.465,-727.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,1.0,1620000.0,0.0,0.0,0.0,0.0,16.0,1007153.415,1007153.415,-12.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,66.0,806127.975,835985.340,-14.0


In [ ]:
dados.to_csv(RAW_DATA_DIR / "conjunto_completo.csv", index=False)